In [ ]:
from theia.terrain import SrtmTerrainModel


terrain = SrtmTerrainModel()
terrain.elevationAt(46.801111, 8.226667)

In [ ]:
%%timeit
terrain.elevationAt(46.801111, 8.226667)

In [ ]:
%%timeit
terrain.elevationAt(46.801111, 8.226667)

In [ ]:
import math

from theia.config import ELEVATION_DATA_DIR
from theia.terrain import interpolate_elevation_tile, load_hgt_file


def elevationAt(lat: float, lon: float) -> float:
    lat0 = math.floor(lat)
    lon0 = math.floor(lon)

    ns = "N" if lat0 >= 0 else "S"
    ew = "E" if lon0 >= 0 else "W"
    filename = f"{ns}{abs(lat0):02d}{ew}{abs(lon0):03d}.hgt"

    arr = load_hgt_file(f"{ELEVATION_DATA_DIR}/{filename}")

    # local fractional degree within tile
    lat_f = lat - lat0
    lon_f = lon - lon0

    return interpolate_elevation_tile(lat_f, lon_f, arr)


def elevationAt_new(lat: float, lon: float) -> float:
    lat0 = math.floor(lat)
    lon0 = math.floor(lon)

    print(lat0)
    print(lon0)

    ns = "N" if lat0 >= 0 else "S"
    ew = "E" if lon0 >= 0 else "W"

    print(ns)
    print(ew)
    filename = f"{ns}{abs(lat0):02d}{ew}{abs(lon0):03d}.hgt"
    print(filename)

    arr = load_hgt_file(f"{ELEVATION_DATA_DIR}/{filename}")

    # local fractional degree within tile
    lat_f = lat - lat0
    lon_f = lon - lon0

    return interpolate_elevation_tile(lat_f, lon_f, arr)


elevationAt(46.801111, 8.226667)
elevationAt_new(46.801111, 8.226667)

In [ ]:
from theia.terrain import interpolate_elevation_tile, load_hgt_file

load_hgt_file(46, 8)

In [ ]:
import timeit

import numpy as np

N = 1_000_000
times = timeit.repeat("arr = load_hgt_file(f'{ELEVATION_DATA_DIR}/{filename}')", globals={"ELEVATION_DATA_DIR": ELEVATION_DATA_DIR, "filename": "N46E008.hgt", "load_hgt_file": load_hgt_file}, number=N, repeat=7)
# times_new = timeit.repeat("x.__floor__()", globals={"math": math, "x": 46.801111}, number=N, repeat=7)

print(f"Original: {np.mean(times) / N * 1e9:.1f} +/- {np.std(times) / N * 1e9:.1f} ns")
# print(f"New:      {np.mean(times_new) / N * 1e9:.1f} +/- {np.std(times_new) / N * 1e9:.1f} ns")

In [ ]:
import timeit

import numpy as np

N = 1_000_000
times = timeit.repeat("f'{ns}{abs(lat0):02d}{ew}{abs(lon0):03d}.hgt'", globals={"lat0": 46, "lon0": 8, "ns": "N", "ew": "E"}, number=N, repeat=7)
# times_new = timeit.repeat("x.__floor__()", globals={"math": math, "x": 46.801111}, number=N, repeat=7)

print(f"Original: {np.mean(times) / N * 1e9:.1f} +/- {np.std(times) / N * 1e9:.1f} ns")
# print(f"New:      {np.mean(times_new) / N * 1e9:.1f} +/- {np.std(times_new) / N * 1e9:.1f} ns")

In [ ]:
%%timeit
elevationAt_new(46.801111, 8.226667)

In [ ]:
%%timeit
elevationAt(46.801111, 8.226667)

In [ ]:
import cProfile


profiler = cProfile.Profile()
profiler.enable()

for _ in range(1_000_000):
    terrain.elevationAt(46.801111, 8.226667)

profiler.disable()
profiler.dump_stats("profile_elevation.prof")

In [ ]:
import math

import numba
from numpy import radians
import numpy as np

from theia.distance import R_EARTH, haversine
from theia.terrain import AbstractTerrainModel
from theia.types import Point


def has_line_of_sight_ray_marching(
    p1: Point,
    p2: Point,
    terrain_model: AbstractTerrainModel,
    step_m: float,
) -> float:
    """
    Checks line of sight between two points accounting for Earth curvature
    and terrain elevation.

    step_m controls sampling resolution along the path.
    """

    def rad(p):
        return math.radians(p.lat), math.radians(p.lon)

    lat1, lon1 = rad(p1)
    lat2, lon2 = rad(p2)

    distance = haversine(p1.lon, p1.lat, p2.lon, p2.lat)

    if distance == 0:
        return True

    steps = max(1, int(distance / step_m) + 1)

    # Heights above Earth's center
    h1 = R_EARTH + p1.alt
    h2 = R_EARTH + p2.alt

    for i in range(1, steps):
        t = i / steps

        # Interpolate along great circle
        A = math.sin((1 - t) * distance / R_EARTH) / math.sin(distance / R_EARTH)
        B = math.sin(t * distance / R_EARTH) / math.sin(distance / R_EARTH)

        x = A * math.cos(lat1) * math.cos(lon1) + B * math.cos(lat2) * math.cos(lon2)
        y = A * math.cos(lat1) * math.sin(lon1) + B * math.cos(lat2) * math.sin(lon2)
        z = A * math.sin(lat1) + B * math.sin(lat2)

        lat = math.atan2(z, math.sqrt(x * x + y * y))
        lon = math.atan2(y, x)

        lat_deg = math.degrees(lat)
        lon_deg = math.degrees(lon)

        # Terrain height above Earth's center
        terrain = R_EARTH + terrain_model.elevationAt(lat_deg, lon_deg)

        # Height of LOS ray at this point
        ray_height = (1 - t) * h1 + t * h2

        if terrain > ray_height:
            return False

    return True


def has_line_of_sight_ray_marching_new(
    p1: Point,
    p2: Point,
    terrain_model: AbstractTerrainModel,
    step_m: float,
) -> float:
    """
    Checks line of sight between two points accounting for Earth curvature
    and terrain elevation.

    step_m controls sampling resolution along the path.
    """
    lat1 = math.radians(p1.lat)
    lon1 = math.radians(p1.lon)
    lat2 = math.radians(p2.lat)
    lon2 = math.radians(p2.lon)

    distance = haversine(p1.lon, p1.lat, p2.lon, p2.lat)

    if distance == 0:
        return True

    steps = max(1, int(distance / step_m) + 1)

    # Heights above Earth's center
    h1 = R_EARTH + p1.alt
    h2 = R_EARTH + p2.alt

    for i in range(1, steps):
        t = i / steps

        # Interpolate along great circle
        A = math.sin((1 - t) * distance / R_EARTH) / math.sin(distance / R_EARTH)
        B = math.sin(t * distance / R_EARTH) / math.sin(distance / R_EARTH)

        x = A * math.cos(lat1) * math.cos(lon1) + B * math.cos(lat2) * math.cos(lon2)
        y = A * math.cos(lat1) * math.sin(lon1) + B * math.cos(lat2) * math.sin(lon2)
        z = A * math.sin(lat1) + B * math.sin(lat2)

        lat = math.atan2(z, math.sqrt(x * x + y * y))
        lon = math.atan2(y, x)

        lat_deg = math.degrees(lat)
        lon_deg = math.degrees(lon)

        # Terrain height above Earth's center
        terrain = R_EARTH + terrain_model.elevationAt(lat_deg, lon_deg)

        # Height of LOS ray at this point
        ray_height = (1 - t) * h1 + t * h2

        if terrain > ray_height:
            return False

    return True


@numba.njit
def has_line_of_sight_ray_marching_new_numba(
    p1: Point,
    p2: Point,
    terrain_model: AbstractTerrainModel,
    step_m: float,
) -> float:
    """
    Checks line of sight between two points accounting for Earth curvature
    and terrain elevation.

    step_m controls sampling resolution along the path.
    """
    lat1 = math.radians(p1.lat)
    lon1 = math.radians(p1.lon)
    lat2 = math.radians(p2.lat)
    lon2 = math.radians(p2.lon)

    distance = haversine(p1.lon, p1.lat, p2.lon, p2.lat)

    if distance == 0:
        return True

    steps = max(1, int(distance / step_m) + 1)

    # Heights above Earth's center
    h1 = R_EARTH + p1.alt
    h2 = R_EARTH + p2.alt

    for i in range(1, steps):
        t = i / steps

        # Interpolate along great circle
        A = math.sin((1 - t) * distance / R_EARTH) / math.sin(distance / R_EARTH)
        B = math.sin(t * distance / R_EARTH) / math.sin(distance / R_EARTH)

        x = A * math.cos(lat1) * math.cos(lon1) + B * math.cos(lat2) * math.cos(lon2)
        y = A * math.cos(lat1) * math.sin(lon1) + B * math.cos(lat2) * math.sin(lon2)
        z = A * math.sin(lat1) + B * math.sin(lat2)

        lat = math.atan2(z, math.sqrt(x * x + y * y))
        lon = math.atan2(y, x)

        lat_deg = math.degrees(lat)
        lon_deg = math.degrees(lon)

        # Terrain height above Earth's center
        terrain = R_EARTH + terrain_model.elevationAt(lat_deg, lon_deg)

        # Height of LOS ray at this point
        ray_height = (1 - t) * h1 + t * h2

        if terrain > ray_height:
            return False

    return True


In [ ]:
from theia.terrain import SrtmTerrainModel


lat1 = 45.7
lat2 = 45.9
lon1 = 5.7
lon2 = 10.6

terrain = SrtmTerrainModel()

p1 = Point(
    lat=lat1,
    lon=lon1,
    alt=terrain.elevationAt(lat1, lon1) + 10.0,
)
p2 = Point(
    lat=lat2,
    lon=lon2,
    alt=terrain.elevationAt(lat2, lon2) + 10.0,
)

has_line_of_sight_ray_marching(p1, p2, terrain, 30)
has_line_of_sight_ray_marching_new(p1, p2, terrain, 30)
# has_line_of_sight_ray_marching_new_numba(p1, p2, terrain, 30)

In [ ]:
%%timeit
has_line_of_sight_ray_marching(p1, p2, terrain, 30)

In [ ]:
%%timeit
has_line_of_sight_ray_marching_new(p1, p2, terrain, 30)

In [ ]:
%%timeit
haversine_orig(lon1, lat1, lon2, lat2)

In [ ]:
import timeit
import math
import numpy as np

import numba


@numba.njit
def f(x: float):
    return np.rad2deg(x)


@numba.njit
def g(x: float):
    return math.degrees(x)


x = 1.5707963  # ~pi/2

# Warmup
math.degrees(x)
np.rad2deg(x)
f(x)
g(x)

N = 1_000_000

math_times = timeit.repeat(
    "math.sin(x)", globals={"math": math, "x": x}, number=N, repeat=7
)
numpy_times = timeit.repeat("np.sin(x)", globals={"np": np, "x": x}, number=N, repeat=7)
# math_numba_times = timeit.repeat('g(x)', globals={'np': np, 'x': x, 'g': g}, number=N, repeat=7)
# numpy_numba_times = timeit.repeat('f(x)', globals={'np': np, 'x': x, 'f': f}, number=N, repeat=7)

import statistics

math_ns = [t / N * 1e9 for t in math_times]
numpy_ns = [t / N * 1e9 for t in numpy_times]
# math_numba_ns = [t/N*1e9 for t in math_numba_times]
# numpy_numba_ns = [t/N*1e9 for t in numpy_numba_times]

print(
    f"math.sin:  {statistics.mean(math_ns):.1f} +/- {statistics.stdev(math_ns):.1f} ns"
)
print(
    f"np.sin:    {statistics.mean(numpy_ns):.1f} +/- {statistics.stdev(numpy_ns):.1f} ns"
)
# print(f'math.degrees + numba:    {statistics.mean(math_numba_ns):.1f} +/- {statistics.stdev(math_numba_ns):.1f} ns')
# print(f'np.rad2deg + numba:    {statistics.mean(numpy_numba_ns):.1f} +/- {statistics.stdev(numpy_numba_ns):.1f} ns')
print(
    f"Ratio:         {statistics.mean(numpy_ns) / statistics.mean(math_ns):.1f}x slower"
)

In [ ]:
%%timeit
math.degrees(np.pi)

In [ ]:
%%timeit
np.rad2deg(np.pi)

In [ ]:
import numpy as np

from theia.simulation.factories.performance_demo import PerformanceDemoFactory

rng = np.random.default_rng(seed=425389)

sensors = PerformanceDemoFactory(rng)._get_pcl_sensors()

In [ ]:
from theia.grids import LatLonHeightGrid, LatLonTerrainGrid
from theia.terrain import SrtmTerrainModel


grid = LatLonHeightGrid(
    lat_start=46.88835,
    lat_stop=47.69682,
    lat_res=0.01,
    lon_start=7.46548,
    lon_stop=8.76724,
    lon_res=0.01,
    height_start=1000,
    height_stop=1000,
    height_res=1.0,
    terrain_model=SrtmTerrainModel(),
)

In [ ]:
from theia.detection.pcl import PclDetector

rcs_grids = []
for sensor in sensors:
    rcs_grids.append(
        PclDetector().minimum_detectable_rcs_grid(
            sensor.receiver, sensor.transmitter, grid
        )
    )

In [ ]:
from matplotlib import pyplot as plt
import matplotlib.colors as mcolors

# Define your custom unequal boundaries
bounds = [0, 15]  # ← your values here

# Create a discrete norm from those boundaries
norm = mcolors.BoundaryNorm(boundaries=bounds, extend="max", ncolors=256)

fig, axes = plt.subplots(2, 2, figsize=(16, 9))

for ax, rcs_grid in zip(axes.flatten(), rcs_grids, strict=True):
    img = ax.imshow(rcs_grid[:, :, 0], norm=norm, cmap="viridis")
fig.colorbar(img)

In [ ]:
n_detections = np.sum(np.stack(rcs_grids, axis=-1)[:, :, 0, :] < 1, axis=-1)

In [ ]:
from theia.detection.pcl import pcl_track_init_update_masks


init_mask, update_mask = pcl_track_init_update_masks(
    PclDetector(),
    sensors,
    grid,
    1.0,
)

In [ ]:
from theia.util import mask_to_polygon


mask_to_polygon(
    n_detections >= 3,
    grid.latitude_values[0],
    grid.lat_res,
    grid.longitude_values[0],
    grid.lon_res,
)[0]

In [ ]:
plt.imshow(init_mask)

In [ ]:
plt.imshow((n_detections >= 3))

In [ ]:
plt.imshow((n_detections == 2))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

norm = mcolors.BoundaryNorm(
    boundaries=[0, 0.5, 1.5, 2.5, 3.5], extend="max", ncolors=256
)
img = ax.imshow(n_detections.astype(int), norm=norm)
ax.invert_yaxis()
cbar = fig.colorbar(img)
cbar.set_ticks([0, 1, 2, 3])

# Local terrain maxima for transmitter positions

In [ ]:
import pandas as pd

from theia.grids import LatLonTerrainGrid
from theia.terrain import SrtmTerrainModel


grid = LatLonTerrainGrid(
    lat_start=46.88835,
    lat_stop=47.69682,
    lat_res=0.001,
    lon_start=7.46548,
    lon_stop=8.76724,
    lon_res=0.001,
    terrain_model=SrtmTerrainModel(),
)

# init_mask, update_mask = pcl_track_init_update_masks(
#     PclDetector(),
#     sensors,
#     grid,
#     1.0,
# )


points = grid.get_points_local_maxima(radius=100)

tx_locs = points[[38, 42, 47, 58, 54], :]

In [ ]:
points[50, :]

In [ ]:
tx_locs

In [ ]:
points.shape

In [ ]:
import datetime

from theia.config import SIDC_RED_FIXED_WING
from theia.maneuvers import ConstantSpeedStraightManeuver
from theia.types import Point

terrain = SrtmTerrainModel()
trajectory = ConstantSpeedStraightManeuver(
    stop_point=Point(
        lat=47.2240,
        lon=9.7335,
        alt=terrain.elevationAt(47.2240, 9.7335),
    ),
    speed=300.0,
).to_trajectory(
    start_time=datetime.datetime.now(),
    start_pos=Point(
        lat=47.6100,
        lon=7.4212,
        alt=terrain.elevationAt(47.6100, 7.4212),
    ),
    target_id=1,
    sidc=SIDC_RED_FIXED_WING,
    rcs=1.0,
)
targets = trajectory.sample_in_time(datetime.timedelta(seconds=20))

In [ ]:
import numpy as np
from theia.test_data import build_pcl_receiver
from theia.types import Polarization, Transmitter
from theia.util import to_dB

tx_locs = np.array(
    [
        [47.42035, 7.95848, 954.34130859],
        [47.47835, 7.64948, 756.82000732],
        [47.53035, 8.17048, 697.52001953],
        [47.68535, 8.29048, 704.26000977],
    ]
)

txs = []
for i, (lat, lon, alt) in enumerate(tx_locs):
    p = Point(
        lat=lat,
        lon=lon,
        alt=alt,
    )
    tx = Transmitter(
        id=i,
        point=p,
        power=10_000,
        erp=to_dB(10_000),
        antenna_height=8,
        antenna_diameter=2,
        antenna_gain=0,
        frequency=95.0,
        pulse_width=0.0,
        bandwidth=0.190,
        polarization=Polarization.VERTICAL,
    )
    txs.append(tx)


rx = build_pcl_receiver(
    rx_id=0,
    point=Point(
        lat=47.42535,
        lon=8.19448,
        alt=636.02301025,
    ),
)

In [ ]:
import folium


map = folium.Map(location=(47.3222, 8.0914), zoom_start=10)
folium.LatLngPopup().add_to(map)
for i, p in enumerate(points):
    folium.Marker((p[0], p[1]), tooltip=f"Point {i}").add_to(map)
for p in tx_locs:
    folium.Marker(
        (p[0], p[1]), tooltip=f"Point {i}", icon=folium.Icon(color="purple")
    ).add_to(map)
for target in targets:
    folium.Marker((target.lat, target.lon), icon=folium.Icon(color="red")).add_to(map)
map

In [ ]:
import datetime

from theia.config import SIDC_RED_FIXED_WING
from theia.data_loading import load_bakom_ukw_transmitters
from theia.maneuvers import ConstantSpeedStraightManeuver
from theia.optimization.optimize_pcl import optimize_pcl_receiver
from theia.terrain import SrtmTerrainModel
from theia.test_data import build_single_target_from_Bodensee
from theia.types import Point

terrain = SrtmTerrainModel()
trajectory = ConstantSpeedStraightManeuver(
    stop_point=Point(
        lat=47.2240,
        lon=9.7335,
        alt=terrain.elevationAt(47.2240, 9.7335),
    ),
    speed=300.0,
).to_trajectory(
    start_time=datetime.datetime.now(),
    start_pos=Point(
        lat=47.6100,
        lon=7.4212,
        alt=terrain.elevationAt(47.6100, 7.4212),
    ),
    target_id=1,
    sidc=SIDC_RED_FIXED_WING,
    rcs=1.0,
)
targets = trajectory.sample_in_time(datetime.timedelta(seconds=20))

roi_lat_min = 47.12621
roi_lon_min = 6.48808
roi_lat_max = 47.86846
roi_lon_max = 8.25671

txs = load_bakom_ukw_transmitters()
txs = [tx for tx in txs if tx.power >= 1000]
txs_roi = [
    tx
    for tx in txs
    if roi_lat_min <= tx.lat <= roi_lat_max and roi_lon_min <= tx.lon <= roi_lon_max
]


In [ ]:
rx, optimizer = optimize_pcl_receiver(
    targets,
    txs_roi,
    roi_lat_min,
    roi_lat_max,
    roi_lon_min,
    roi_lon_max,
)


In [ ]:
x_obs = optimizer.space.params
y_obs = optimizer.space.target

In [ ]:
from matplotlib import pyplot as plt


fig, ax = plt.subplots(figsize=(8, 4.5))

ax.plot(optimizer.space.target)
ax.axhline(15)

In [ ]:
import folium

from theia.mapping import RadarMap


map = RadarMap().to_map()
folium.Marker(
    (rx.lat, rx.lon),
    icon=folium.Icon(color="darkgreen"),
    tooltip=f"Lat: {rx.lat}<br>Lon: {rx.lon}",
).add_to(map)
for tx in txs_roi:
    folium.Marker((tx.lat, tx.lon), tooltip=f"Tx ID = {tx.id}").add_to(map)
for target in targets:
    folium.Marker((target.lat, target.lon), icon=folium.Icon(color="red")).add_to(map)
map

In [ ]:
47.0658, 8.5250

In [ ]:
optimizer.logger

In [ ]:
import numpy as np
import pydantic

from theia.coordinates import POSITIONS_OF_INTEREST, CoordinateTransformations
from theia.terrain import FlatEarthTerrainModel, PlateauHill
from theia.types import Point


class FlatEarthCoordinateSystem(pydantic.BaseModel):
    origin: Point
    terrain_model: FlatEarthTerrainModel

    def model_post_init(self, context):
        self._origin_ecef = np.array(
            CoordinateTransformations.geodetic_to_cartesian(
                self.origin.lat,
                self.origin.lon,
                self.origin.alt,
            )
        )

    def flat_earth_to_ecef(
        self,
        x: float,
        y: float,
        z: float,
    ) -> tuple[float, float, float]:
        R = np.array(self.terrain_model.plane_directions)
        print(np.max(np.abs(R @ R.T - np.eye(3))))
        print(np.max(np.abs(R.T @ R - np.eye(3))))
        p_ecef = R.T @ np.array((x, y, z)) + self._origin_ecef
        return tuple(p_ecef)

    def flat_earth_to_geodetic(
        self,
        x: float,
        y: float,
        z: float,
    ) -> Point:
        p_ecef = self.flat_earth_to_ecef(x, y, z)
        p = CoordinateTransformations.cartesian_to_geodetic(*p_ecef)
        return Point(
            lat=p[0],
            lon=p[1],
            alt=p[2],
        )


def build_flat_coord_wall(
    bottom_left: tuple[float, float],
    top_right: tuple[float, float],
    height: float,
    coordinate_system: FlatEarthCoordinateSystem,
) -> PlateauHill:
    p1 = coordinate_system.flat_earth_to_geodetic(bottom_left[0], bottom_left[1], 0)
    p2 = coordinate_system.flat_earth_to_geodetic(top_right[0], top_right[1], 0)

    lat_min = min(p1.lat, p2.lat)
    lat_max = max(p1.lat, p2.lat)
    lon_min = min(p1.lon, p2.lon)
    lon_max = max(p1.lon, p2.lon)

    return PlateauHill(
        lat_min=lat_min,
        lat_max=lat_max,
        lon_min=lon_min,
        lon_max=lon_max,
        height=height,
    )


In [ ]:
from theia.terrain import FlatEartWithHillsTerrainModel

flat_terrain = FlatEarthTerrainModel()

origin_lat = POSITIONS_OF_INTEREST["CH_CENTER"]["lat"]
origin_lon = POSITIONS_OF_INTEREST["CH_CENTER"]["lon"]
origin_alt = flat_terrain.elevationAt(origin_lat, origin_lon)
origin = Point(lat=origin_lat, lon=origin_lon, alt=origin_alt)

extent_lat = 8.0
extent_lon = 8.0

system = FlatEarthCoordinateSystem(origin=origin, terrain_model=flat_terrain)

plateau1 = build_flat_coord_wall(
    (-200_000, -75_000),
    (
        -198_000,
        425_000,
    ),
    1000,
    system,
)
plateau2 = build_flat_coord_wall(
    (-200_000, -75_000),
    (500_000, -73_000),
    1000,
    system,
)

terrain = FlatEartWithHillsTerrainModel(plateaus=[plateau1])

# Sanity check.
assert np.allclose(
    system.flat_earth_to_geodetic(0, 0, 0).as_tuple(),
    origin.as_tuple(),
)

In [ ]:
p_monostatic1_flat = (0, 0, 0)
p_monostatic2_flat = (200_000, 0, 0)
p_monostatic3_flat = (0, 100_000, 0)

p_monostatic1 = system.flat_earth_to_geodetic(*p_monostatic1_flat)
p_monostatic2 = system.flat_earth_to_geodetic(*p_monostatic2_flat)
p_monostatic3 = system.flat_earth_to_geodetic(*p_monostatic3_flat)

In [ ]:
p_monostatic1 = system.flat_earth_to_ecef(*p_monostatic1_flat)
p_monostatic2 = system.flat_earth_to_ecef(*p_monostatic2_flat)
p_monostatic3 = system.flat_earth_to_ecef(*p_monostatic3_flat)

points = np.array((p_monostatic1, p_monostatic2, p_monostatic3))

d1 = points[1, :] - points[0, :]
d2 = points[2, :] - points[0, :]

np.sqrt(abs(np.dot(d1, d2)))

# import plotly.express as px
# px.scatter_3d(x=points[:, 0], y=points[:, 1], z=points[:, 2])

In [ ]:
from theia.export_paraview import ParaviewExporter, PointOfInterest


rotation_center = np.mean(np.array(terrain.corners_ecef), axis=0)
rotation_center = Point(
    lat=rotation_center[0],
    lon=rotation_center[1],
    alt=rotation_center[2],
)

exporter = ParaviewExporter(
    "output/terrain_with_hills",
    lat_min=origin.lat - extent_lat / 2.0,
    lat_max=origin.lat + extent_lat / 2.0,
    lat_res=0.01,
    lon_min=origin.lon - extent_lon / 2.0,
    lon_max=origin.lon + extent_lon / 2.0,
    lon_res=0.01,
    terrain_model=terrain,
    elevation_factor=10.0,
    fill_negative_alts=False,
)
exporter.export(
    [],
    [
        PointOfInterest(
            id=0,
            label="Radar1",
            type="Rx",
            lat=p.lat,
            lon=p.lon,
            alt=p.alt,
        )
        for p in [p_monostatic1, p_monostatic2, p_monostatic3]
    ],
)

In [ ]:
from theia.radar_equation import calculate_maximum_monostatic_range
from theia.test_data import get_uetliberg_radar


radar = get_uetliberg_radar(power=2000, frequency=1500, rx_bandwidth=2, diameter=2)
calculate_maximum_monostatic_range(radar, 1.0) / 1e3

In [ ]:
from theia.terrain import PlateauHill


In [ ]:
from theia.export_paraview import ParaviewExporter
from theia.terrain import ConstantSphereTerrainModel


terrain = ConstantSphereTerrainModel(alt=0)

exporter = ParaviewExporter(
    "output/sphere",
    lat_min=34.01624,
    lat_max=55.97380,
    lat_res=0.1,
    lon_min=-11.85964,
    lon_max=38.14561,
    lon_res=0.1,
    terrain_model=terrain,
    elevation_factor=1.0,
)
exporter.export([], [])

In [ ]:
from theia.export_paraview import ParaviewExporter
from theia.terrain import (
    FlatEartWithHillsTerrainModel,
    FlatEarthTerrainModel,
    PlateauHill,
)


# terrain = FlatEarthTerrainModel()
terrain = FlatEartWithHillsTerrainModel(
    plateaus=[
        PlateauHill(
            lat_min=40.0,
            lat_max=45.0,
            lon_min=0.0,
            lon_max=5.0,
            height=10_000,
        )
    ]
)

exporter = ParaviewExporter(
    "./output/flat",
    lat_min=34.01624,
    lat_max=55.97380,
    lat_res=0.01,
    lon_min=-11.85964,
    lon_max=38.14561,
    lon_res=0.01,
    terrain_model=terrain,
    elevation_factor=1.0,
    fill_negative_alts=False,
)
exporter.export([], [])

In [ ]:
from theia.coordinates import CoordinateTransformations


CoordinateTransformations.cartesian_to_geodetic(
    3500900.3133147047, -735180.1903229321, 5262810.433892342
)

In [ ]:
import numpy as np

from theia.distance import R_EARTH


R_EARTH - np.linalg.norm(
    np.array(
        [3500900.3133147047, -735180.1903229321, 5262810.433892342], dtype=np.float128
    )
)

In [ ]:
(
    np.square(3500900.3133147047) / 1e13,
    np.square(735180.1903229321) / 1e13,
    np.square(5262810.433892342) / 1e13,
)

In [ ]:
from functools import cached_property

import numpy as np

from theia.coordinates import CoordinateTransformations
from theia.distance import R_EARTH
from theia.terrain import AbstractTerrainModel
from theia.types import Point


a


In [ ]:
from theia.plotting import plot_profile
from theia.terrain import ConstantSphereTerrainModel, SrtmTerrainModel
from theia.types import Point

terrain = SrtmTerrainModel()
terrain = ConstantSphereTerrainModel(alt=100)

plot_profile(
    Point(lat=47.29413, lon=8.28446, alt=terrain.elevationAt(47.29413, 8.28446)),
    Point(lat=46.01222, lon=7.15267, alt=terrain.elevationAt(46.01222, 7.15267)),
    terrain_model=terrain,
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from theia.ellipsoid import Ellipsoid, EllipsoidIntersection

e1 = Ellipsoid(
    p1=(4282300.676063238, 639387.6735549602, 4668806.332152337),
    p2=(4278369.997370845, 644603.2701877941, 4671416.235940153),
    r=27809.251860388176,
)
e2 = Ellipsoid(
    p1=(4282300.676063238, 639387.6735549602, 4668806.332152337),
    p2=(4281690.458270983, 629501.0957207535, 4670408.211252162),
    r=18720.50853567265,
)
ei = EllipsoidIntersection(e1=e1, e2=e2, sigma_r1=0, sigma_r2=0)


rng = np.random.default_rng(seed=830752)
pe1 = np.vstack([e1.sample_surface_uniformly(rng) for _ in range(500)])
pe2 = np.vstack([e2.sample_surface_uniformly(rng) for _ in range(500)])
pi = np.vstack([ei.sample(rng, n_batch=50) for _ in range(1000)])

fig = go.Figure()
fig.add_trace(go.Scatter3d(x=pe1[:, 0], y=pe1[:, 1], z=pe1[:, 2], mode="markers"))
fig.add_trace(go.Scatter3d(x=pe2[:, 0], y=pe2[:, 1], z=pe2[:, 2], mode="markers"))
fig.add_trace(go.Scatter3d(x=pi[:, 0], y=pi[:, 1], z=pi[:, 2], mode="markers"))
fig.update_traces(marker_size=1)

In [ ]:
import time


ei = EllipsoidIntersection(e1=e1, e2=e2, sigma_r1=0, sigma_r2=0)

N = 100

times = []
for n_batch in [50, 100, 200, 250, 300, 400, 500]:
    ei = EllipsoidIntersection(e1=e1, e2=e2, sigma_r1=0, sigma_r2=0)
    start = time.perf_counter()
    s = [ei.sample(rng, n_batch=n_batch) for _ in range(N)]
    stop = time.perf_counter()
    times.append(
        {
            "n_batch": n_batch,
            "time": (stop - start) / N,
            "cold_start": True,
        }
    )

    start = time.perf_counter()
    s = [ei.sample(rng, n_batch=n_batch) for _ in range(N)]
    stop = time.perf_counter()
    times.append(
        {
            "n_batch": n_batch,
            "time": (stop - start) / N,
            "cold_start": False,
        }
    )

In [ ]:
import pandas as pd
import seaborn as sns


df = pd.DataFrame(times)
sns.barplot(data=df, x="n_batch", y="time", orient="v", hue="cold_start")

In [ ]:
%%timeit
s = ei.sample(rng, n_batch=100)

In [ ]:
# Idea:
# Place PET receivers so that they have as much line of sight to the
# target trajectory as possible. That means:
# - Sample target waypoints.
# - For each waypoint: Calculate visible points on lat-lon-terrain-grid (binary mask)
# - Sum up the masks for each waypoint.
# - Place sensors at maxima.

# TODO:
# Replace sidc code by "TargetCategory", which has an "archetype" and a SIDC code.
# Open for extension in the future.

In [ ]:
from theia.coordinates import POSITIONS_OF_INTEREST


lat = POSITIONS_OF_INTEREST["CH_CENTER"]["lat"]
lon = POSITIONS_OF_INTEREST["CH_CENTER"]["lon"]

In [ ]:
from theia.terrain import elevationAt


elevationAt(lat, lon)

In [ ]:
import abc
import math

from theia.config import ELEVATION_DATA_DIR
from theia.terrain import interpolate_elevation_tile, load_hgt_file
from theia.types import Point


model = SrtmTerrainModel()

In [ ]:
%%timeit
alt = elevationAt(lat, lon)

In [ ]:
%%timeit
alt = model.elevationAt(lat, lon)

In [ ]:
import numba
import numpy as np


@numba.njit(cache=True, inline="always")
def _norm3(v):
    return np.sqrt(v[0] * v[0] + v[1] * v[1] + v[2] * v[2])


@numba.njit(cache=True, inline="always")
def _orthonormal_pair(e):
    """Two unit vectors perpendicular to unit vector e."""
    if abs(e[0]) < 0.9:
        t = np.array([1.0, 0.0, 0.0])
    else:
        t = np.array([0.0, 1.0, 0.0])
    # e2 = t − (t·e)e  normalised
    dot = t[0] * e[0] + t[1] * e[1] + t[2] * e[2]
    e2 = np.array([t[0] - dot * e[0], t[1] - dot * e[1], t[2] - dot * e[2]])
    e2 /= _norm3(e2)
    # e3 = e × e2
    e3 = np.array(
        [
            e[1] * e2[2] - e[2] * e2[1],
            e[2] * e2[0] - e[0] * e2[2],
            e[0] * e2[1] - e[1] * e2[0],
        ]
    )
    return e2, e3


@numba.njit(cache=True)
def _sample_on_ellipsoid(T, R, brange):
    """
    Sample one point uniformly on the prolate-spheroid surface
    defined by foci T, R and bistatic range brange.
    """
    a = 0.5 * brange
    fv = R - T
    c = 0.5 * _norm3(fv)  # focal half-distance

    if c >= a:  # degenerate (invalid range)
        return 0.5 * (T + R)

    b2 = a * a - c * c
    b = np.sqrt(b2)
    a2 = a * a

    e = fv / (2.0 * c)  # unit major-axis vector
    e2, e3 = _orthonormal_pair(e)
    ctr = 0.5 * (T + R)

    # Rejection sampling for surface-uniform (θ, φ)
    while True:
        cos_t = 2.0 * np.random.random() - 1.0  # uniform-on-sphere proposal
        sin_t = np.sqrt(max(0.0, 1.0 - cos_t * cos_t))
        phi = 2.0 * np.pi * np.random.random()

        # Accept prob = sqrt(b²cos²θ + a²sin²θ) / a
        acc = np.sqrt(b2 * cos_t * cos_t + a2 * sin_t * sin_t) / a
        if np.random.random() <= acc:
            cp = np.cos(phi)
            sp = np.sin(phi)
            x = ctr + (a * cos_t) * e + (b * sin_t * cp) * e2 + (b * sin_t * sp) * e3
            return x

In [ ]:
%%timeit
_sample_on_ellipsoid(np.array((0, 0, 0)), np.array((4000, 0, 0)), 150_000)

In [ ]:
from theia.ellipsoid import Ellipsoid


ellipsoid = Ellipsoid((0, 0, 0), (4000, 0, 0), 150_000)

In [ ]:
%%timeit
points = ellipsoid.sample_surface(1, 1)

In [ ]:
# import datetime

# from theia.detection.pet import suggest_pet_receiver_locations
# from theia.grids import LatLonTerrainGrid
# from theia.test_data import build_single_target_from_Bodensee

# trajectory = build_single_target_from_Bodensee(target_id=0)
# targets = trajectory.sample_in_time(datetime.timedelta(seconds=20))
# target_positions = [t.point for t in targets]

# grid = LatLonTerrainGrid(
#     lat_start=46.78125,
#     lat_stop=48.12577,
#     lon_start=7.17484,
#     lon_stop=9.49275,
#     lat_res=0.01,
#     lon_res=0.01,
# )

# best_points = suggest_pet_receiver_locations(grid, target_positions)

In [ ]:
from theia.config import SIDC_RED_FIXED_WING
from theia.simulation.controllers.waypoint_target_controller import (
    WaypointTargetController,
)
from theia.terrain import elevationAt
from theia.test_data import build_fighter_jet_radar, build_single_target_from_Bodensee
from theia.types import Point, Receiver


def build_pet_only_scenario():
    pet_receiver_positions = [
        Point(lat=47.6888, lon=8.6138, alt=elevationAt(47.6888, 8.6138)),
        Point(lat=47.6888, lon=8.6138, alt=elevationAt(47.6888, 8.6138)),
    ]

    pet_receivers: list[Receiver] = []
    for i, pos in enumerate(pet_receiver_positions):
        rx = build_fighter_jet_radar(0, 0, 0).receiver.model_copy(deep=True)
        rx.point = pos
        rx.id = i
        pet_receivers.append(rx)

    trajectory = build_single_target_from_Bodensee(target_id=0)
    red_controller = WaypointTargetController.from_trajectory(
        trajectory=trajectory,
        name="Emitting target",
        sidc=SIDC_RED_FIXED_WING,
    )

In [ ]:
import folium

from theia.mapping import RadarMap


map = RadarMap().to_map()
for i, point in enumerate(best_points):
    folium.Marker(
        (point[0], point[1]),
        tooltip=f"Point index = {i}<br />Lat={point[0]:.4f} °<br />Lon={point[1]:.4f} °",
    ).add_to(map)
map

In [ ]:
plt.imshow(n_los)

In [ ]:
%%timeit
los = has_line_of_sight(
    target_pos,
    Point(lat=point[0], lon=point[1], alt=point[2]),
    60,
)

In [ ]:
from theia.coverage import calculate_coverage


calculate_coverage

In [ ]:
from theia.simulation.logging import LogLoader


loader = LogLoader("log.json")

In [ ]:
loader.red_monostatic_radar_detections

In [ ]:
import datetime

from theia.coordinates import POSITIONS_OF_INTEREST
from theia.test_data import (
    build_fighter_jet_radar,
    build_flores_monostatic_radar,
    build_single_target_from_Bodensee,
)
from theia.types import Point


trajectory = build_single_target_from_Bodensee()
target = trajectory(
    datetime.datetime(
        year=2026,
        month=4,
        day=29,
        hour=0,
        minute=0,
        second=58,
    )
)
red_sensor = build_fighter_jet_radar(0, 0, 0)
red_sensor.receiver.point = target.point
red_sensor.transmitter.point = target.point
blue_sensor = build_flores_monostatic_radar(
    Point(
        lat=POSITIONS_OF_INTEREST["Uetliberg"]["lat"],
        lon=POSITIONS_OF_INTEREST["Uetliberg"]["lon"],
        alt=POSITIONS_OF_INTEREST["Uetliberg"]["alt"],
    ),
    1,
    1,
    1,
)

In [ ]:
from theia.line_of_sight import has_line_of_sight


has_line_of_sight(red_sensor.receiver.point, blue_sensor.receiver.point, 30)

In [ ]:
from theia.radar_equation import calculate_maximum_monostatic_range


r_max_red = calculate_maximum_monostatic_range(red_sensor, 1.0)
r_max_blue = calculate_maximum_monostatic_range(blue_sensor, 1.0)

In [ ]:
from theia.coverage import calculate_coverage


coverage_red = calculate_coverage(
    red_sensor.receiver.point, r_max_red, blue_sensor.receiver.alt, d_theta=2
)
coverage_blue = calculate_coverage(
    blue_sensor.receiver.point, r_max_blue, 1000.0, d_theta=2
)

In [ ]:
from theia.mapping import RadarMap


RadarMap(
    sensors={
        "RED": red_sensor,
        "BLUE": blue_sensor,
    },
    polygons={
        "Coverage RED": coverage_red,
        # "Coverage BLUE": coverage_blue,
    },
).to_map()

In [ ]:
from theia.plotting import plot_profile


plot_profile(
    target.point,
    blue_sensor.receiver.point,
    "RED sensor",
    "BLUE sensor (target)",
)

In [ ]:
radar = loader.red_monostatic_radars[0]

In [ ]:
%%timeit
res = calculate_coverage(radar.receiver.point, r_max, 0.0)